# Week 45, Day 4: Breadth-first: level order

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w045d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

T = TreeNode
t = T(1, T(2, T(4), T(5)), T(3, None, T(6, T(7))))
queue, out = deque([t]), []
while queue:
    node = queue.popleft()
    out.append(node.val)
    for child in (node.left, node.right):
        if child:
            queue.append(child)
print(out)

In [ ]:
from collections import deque

class Node:
    def __init__(self, value, children=()):
        self.value, self.children = value, list(children)

def level_order(root):
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        out.append(node.value)
        queue.extend(node.children)
    return out

company = Node("CEO", [Node("CTO", [Node("Dev1"), Node("Dev2")]), Node("CFO", [Node("Accounts")]), Node("COO")])
print(level_order(company))

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def levels(root):
    out, queue = [], deque([root] if root else [])
    while queue:
        level = []
        for _ in range(len(queue)):           # exactly the nodes of this level
            node = queue.popleft()
            level.append(node.val)
            for child in (node.left, node.right):
                if child:
                    queue.append(child)
        out.append(level)
    return out

T = TreeNode
t = T(3, T(9), T(20, T(15), T(7)))
print(levels(t), [sum(l) / len(l) for l in levels(t)])

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def widest_level(root):
    best, queue = 0, deque([(root, 0)])
    while queue:
        first, last = queue[0][1], queue[-1][1]
        best = max(best, last - first + 1)
        for _ in range(len(queue)):
            node, pos = queue.popleft()
            if node.left:
                queue.append((node.left, 2 * pos))
            if node.right:
                queue.append((node.right, 2 * pos + 1))
    return best

T = TreeNode
print(widest_level(T(1, T(3, T(5), T(3)), T(2, None, T(9)))))       # 5 _ 3 _ 9: positions 0 to 3

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def min_depth(root):
    """Edges from the root to the nearest leaf, and how many nodes were looked at."""
    queue, looked = deque([(root, 0)]), 0
    while queue:
        node, depth = queue.popleft()
        looked += 1
        if node.left is None and node.right is None:
            return depth, looked
        for child in (node.left, node.right):
            if child:
                queue.append((child, depth + 1))

deep = None
for i in range(2000):
    deep = TreeNode(i, deep)
t = TreeNode("root", TreeNode("leaf"), deep)
print(min_depth(t))

> **Key idea.** Level order visits a tree level by level with a queue: take a node from the front, visit it, add its children at the back. No `seen` set is needed in a tree. To group by level, process exactly `len(queue)` nodes each round (or queue `(node, depth)` pairs). Levels answer views, sums, zigzags and widths; counting gaps, give children positions `2i` and `2i + 1`. Because BFS reaches nodes in order of depth, it can stop at the first leaf it meets. Depth-first uses O(height) memory, breadth-first O(width): choose by the question, then by the tree's shape.

### Exercise: Levels

Write `levels(root)`: a list of lists, one per level, each left to right, using a `deque` and processing one level per round. Then `level_sums(root)` and `deepest_level_values(root)` (the values on the last level).

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def levels(root):
    ___

def level_sums(root):
    ___

def deepest_level_values(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_levels():\n    \"levels\"\n    t = T(3, T(9), T(20, T(15), T(7)))\n    assert levels(t) == [[3], [9, 20], [15, 7]] and level_sums(t) == [3, 29, 22] and deepest_level_values(t) == [15, 7]\n    assert levels(None) == [] and level_sums(None) == [] and deepest_level_values(None) == []\n    tall = None\n    for i in range(3000):\n        tall = T(i, tall)\n    assert len(levels(tall)) == 3000 and deepest_level_values(tall) == [0]")

### Exercise: Views from the side

Write `right_view(root)` and `left_view(root)`: the values you'd see looking at the tree from the right (the last node of each level) and from the left (the first). One breadth-first pass each.

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def right_view(root):
    ___

def left_view(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_views():\n    \"views\"\n    t = T(1, T(2, None, T(5)), T(3, None, T(4)))\n    assert right_view(t) == [1, 3, 4] and left_view(t) == [1, 2, 5]\n    u = T(1, T(2, T(4)), T(3))\n    assert right_view(u) == [1, 3, 4] and left_view(u) == [1, 2, 4] and right_view(None) == []")

### Exercise: Zigzag

Write `zigzag(root)`: the levels as lists, left to right on the first level, right to left on the second, left to right on the third, and so on. Don't change the order in which children join the queue; reverse the lists instead.

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def zigzag(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_zigzag():\n    \"zigzag\"\n    t = T(3, T(9, T(1), T(2)), T(20, T(15), T(7)))\n    assert zigzag(t) == [[3], [20, 9], [1, 2, 15, 7]] and zigzag(None) == [] and zigzag(T(5)) == [[5]]")

### Exercise: The shallowest leaf

Write `min_depth(root)`: the number of edges from the root to the nearest **leaf** (a node with no children), or -1 for an empty tree, with a breadth-first search that stops at the first leaf it meets. The checks build a tree with a leaf near the top and a chain of 50,000 nodes hanging off the other side, and count how many nodes your search reads.

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def min_depth(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\nclass Counted(TreeNode):\n    reads = 0\n    def __getattribute__(self, name):\n        if name == \"left\":\n            Counted.reads += 1\n        return object.__getattribute__(self, name)\ndef test_depth():\n    \"depths\"\n    assert min_depth(T(3, T(9), T(20, T(15), T(7)))) == 1 and min_depth(None) == -1 and min_depth(T(1)) == 0\n    assert min_depth(T(1, T(2, T(3)))) == 2, \"a node with one child isn't a leaf\"\ndef test_early():\n    \"stops early\"\n    chain = None\n    for i in range(50_000):\n        chain = Counted(i, chain)\n    t = Counted(\"root\", Counted(\"leaf\"), chain)\n    Counted.reads = 0\n    assert min_depth(t) == 1 and Counted.reads < 20, f\"{Counted.reads} nodes read: stop at the first leaf\"\n    t2 = Counted(\"root\", Counted(\"a\", Counted(\"leaf\")), chain)\n    Counted.reads = 0\n    assert min_depth(t2) == 2 and Counted.reads < 30")

### Exercise: The widest level

Write `widths(root)`: for each level, its width **counting gaps**: the number of positions from its leftmost to its rightmost node, as if the tree were complete (children of position `p` at `2p` and `2p + 1`). An empty tree gives `[]`.

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def widths(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_widths():\n    \"widths\"\n    assert widths(T(1, T(3, T(5), T(3)), T(2, None, T(9)))) == [1, 2, 4]\n    assert widths(T(1, T(3, T(5, T(6))), T(2, None, T(9, T(7))))) == [1, 2, 4, 7]\n    assert widths(T(1, T(2, T(4)))) == [1, 1, 1] and widths(None) == []\n    left, right = T(0), T(0)\n    root = T(\"r\", left, right)\n    for _ in range(60):\n        left.left = T(0)\n        left = left.left\n        right.right = T(0)\n        right = right.right\n    assert widths(root)[-1] == 2 ** 61")

### Exercise: Levels of a general tree

Day 1's `Node` has a list of children. Write `by_level(root)`: a dictionary from each depth to the list of values at that depth (left to right), and `widest(root)`: the depth with the most nodes (the shallowest, if there's a tie).

In [ ]:
from collections import deque

class Node:
    def __init__(self, value, children=()):
        self.value, self.children = value, list(children)

def by_level(root):
    ___

def widest(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_general():\n    \"general trees\"\n    c = Node(\"CEO\", [Node(\"CTO\", [Node(\"Dev1\"), Node(\"Dev2\"), Node(\"Dev3\")]), Node(\"CFO\", [Node(\"Acc\")]), Node(\"COO\")])\n    assert by_level(c) == {0: [\"CEO\"], 1: [\"CTO\", \"CFO\", \"COO\"], 2: [\"Dev1\", \"Dev2\", \"Dev3\", \"Acc\"]} and widest(c) == 2\n    assert widest(Node(\"a\", [Node(\"b\"), Node(\"c\", [Node(\"d\"), Node(\"e\")])])) == 1 and by_level(Node(\"x\")) == {0: [\"x\"]}")

### Bug hunt: Levels that run together

This tries to group by level, but its inner loop runs `while queue`, so it keeps taking nodes until the queue is empty, including the children it just added: everything ends up in one "level". It prints `[[1, 2, 3, 4, 5]]`. Take exactly one level each round, so it prints `[[1], [2, 3], [4, 5]]`.

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def levels(root):
    out, queue = [], deque([root])
    while queue:
        level = []
        while queue:
            node = queue.popleft()
            level.append(node.val)
            for child in (node.left, node.right):
                if child:
                    queue.append(child)
        out.append(level)
    return out

print(levels(TreeNode(1, TreeNode(2, TreeNode(4)), TreeNode(3, None, TreeNode(5)))))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"one level per round\"\n    expect_output(\"[[1], [2, 3], [4, 5]]\")")

### Bug hunt: A stack in disguise

This "level order" takes nodes from the **back** of the deque, so it's really a depth-first walk: it prints `[1, 3, 5, 2, 4]`. Take them from the front, so it prints `[1, 2, 3, 4, 5]`.

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def level_order(root):
    out, queue = [], deque([root])
    while queue:
        node = queue.pop()
        out.append(node.val)
        for child in (node.left, node.right):
            if child:
                queue.append(child)
    return out

print(level_order(TreeNode(1, TreeNode(2, TreeNode(4)), TreeNode(3, None, TreeNode(5)))))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"from the front\"\n    expect_output(\"[1, 2, 3, 4, 5]\")")

### Bug hunt: Not a leaf

This stops at the first node that's **missing a child**, but a node with one child isn't a leaf: for this tree, it answers 0 (the root has no left child), when the only leaf is two edges down. Stop only at a node with **no** children, so it prints `2`.

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def min_depth(root):
    queue = deque([(root, 0)])
    while queue:
        node, depth = queue.popleft()
        if node.left is None or node.right is None:
            return depth
        for child in (node.left, node.right):
            if child:
                queue.append((child, depth + 1))

print(min_depth(TreeNode(1, None, TreeNode(2, TreeNode(3)))))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"leaves only\"\n    expect_output(\"2\")")

### Puzzle: Grouped level order

Arrange the lines into a function that groups a binary tree's values by level. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# For the tree         1
#                     / \
#                    2   3
#                   /   / \
#                  4   5   6
#                       \
#                        7

p = ___   # the level order, as a list
q = ___   # the right-side view
r = ___   # the zigzag order, as a list of lists
s = ___   # the most nodes in the queue at the start of any round, in the grouped level order
t = ___   # the depth of the shallowest leaf

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [1, 2, 3, 4, 5, 6, 7] and q == [1, 3, 6, 7] and r == [[1], [3, 2], [4, 5, 6], [7]], \"p, q and r\"\n    assert s == 3 and t == 2, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: vertical order

Write `vertical_order(root)`: the tree's values grouped into **columns**. The root is in column 0; a left child is one column to the left of its parent (`col - 1`), a right child one to the right (`col + 1`). Return the columns from leftmost to rightmost, each a list of values ordered by **depth** (top first), and, for nodes at the same depth and column, by value (smallest first). Use breadth-first search with `(node, column, depth)` entries, a dictionary from column to entries, and sort each column. The checks use trees of up to 2,000 nodes.

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def vertical_order(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nT = TreeNode\ndef slow(root):\n    found = []\n    def walk(n, col, depth):\n        if n:\n            found.append((col, depth, n.val))\n            walk(n.left, col - 1, depth + 1)\n            walk(n.right, col + 1, depth + 1)\n    walk(root, 0, 0)\n    cols = sorted(set(c for c, _, _ in found))\n    return [[v for _, _, v in sorted(f for f in found if f[0] == c)] for c in cols]\ndef random_tree(rng, n):\n    root = T(rng.randint(0, 20))\n    nodes = [root]\n    while len(nodes) < n:\n        p = rng.choice(nodes)\n        c = T(rng.randint(0, 20))\n        if p.left is None:\n            p.left = c\n        elif p.right is None:\n            p.right = c\n        else:\n            continue\n        nodes.append(c)\n    return root\ndef test_vertical():\n    \"columns\"\n    assert vertical_order(T(3, T(9), T(20, T(15), T(7)))) == [[9], [3, 15], [20], [7]]\n    assert vertical_order(T(1, T(2, T(4), T(6)), T(3, T(5), T(7)))) == [[4], [2], [1, 5, 6], [3], [7]]\n    assert vertical_order(None) == []\n    rng = random.Random(2)\n    for _ in range(40):\n        t = random_tree(rng, rng.randint(1, 40))\n        assert vertical_order(t) == slow(t)\n    big = random_tree(rng, 2000)\n    assert vertical_order(big) == slow(big)")

## Warm-up

The values come out level by level, left to right: 1; then 2 and 3; then 4, 5 and 6; then 7. Swap `popleft` for `pop` and the same loop becomes a depth-first walk (Week 43): the container decides the order.

## A queue instead of a stack

Level order is breadth-first search on a tree: take the oldest node waiting, visit it, and add its children to the back of the queue. Because a node's children join the queue only after everything already waiting (the rest of its level, and the start of the next), the levels come out in order.

Unlike Week 43's grid search, there's no need for a `seen` set: in a tree every node has one parent, so each node is added to the queue exactly once. For a **general** tree (Day 1's nodes with a list of children), add all the children in order:

## One level at a time

Many questions need to know where one level ends and the next begins. The trick: at the start of each round, the queue holds **exactly one level**, so take `len(queue)` nodes off it, and everything they add is the next level.

`range(len(queue))` is evaluated once, when the `for` loop starts, so the children added during the loop wait for the next round. Another way is to queue `(node, depth)` pairs and start a new level whenever the depth changes; both are common.

## Level questions

Once you have levels, many questions become one line:

| Question | From each level, take... |
|---|---|
| the right-side view (what you'd see looking at the tree from the right) | the last value |
| the left-side view | the first value |
| the level with the largest sum | the sum, and remember the best |
| zigzag order (left to right, then right to left, alternately) | the level, reversed on odd levels |
| the width of each level | the number of nodes, or (counting gaps) the positions of the first and last |

The **width counting gaps** uses Day 2's list positions: number the root 0, a node's children `2i` and `2i + 1` (or `2i + 1` and `2i + 2`), and a level's width is its last position minus its first, plus 1, gaps and all. Python's unlimited integers handle the large positions in deep trees.

## Stopping early

Breadth-first search reaches nodes in order of depth, so the **first** leaf it finds is the shallowest one: it can stop there. A depth-first search would have to explore every branch to be sure.

Two nodes looked at, even though the tree has 2,002. That's the same reason Week 43's BFS found shortest routes: nearest first.

## Breadth or depth?

| | Depth-first (stack or recursion) | Breadth-first (queue) |
|---|---|---|
| order | down one branch at a time | level by level |
| extra memory | O(h): the current path | O(w): the widest level |
| good for | whole-tree answers, paths, sizes, sorted order (in-order) | level questions, nearest or shallowest, shortest paths |

For a balanced tree, h is about log₂ n, but the bottom level holds about half the nodes, so depth-first uses far less memory; for a tall, thin tree it's the other way round. Choose by the question first, then by the tree's shape.

## Practice

## Quiz

## Daily challenge

## Before you go

Take a tree you drew earlier this week and list its levels, its right-side view, and its zigzag order by hand, then check them with the code. Tomorrow: tree problems, a day of classic puzzles that combine everything from this week.